# Performance Analysis: Linked List vs Dynamic Array
**Course:** Data Structures & Algorithms (PR5)  
**Author:** Benedictus Kenneth Setiadi (13224003)

## Executive Summary
Eksperimen ini bertujuan untuk memvalidasi karakteristik performa dari dua arsitektur alokasi memori di bahasa C: **Linked List (Non-contiguous)** dan **Dynamic Array (Contiguous)**. Pengujian ini diotomatisasi menggunakan Python testbench untuk mengukur *Memory Usage*, *Execution Time*, dan *Throughput* pada tiga skala data: Small (1K), Medium (50K), dan Large (200K).

In [7]:
import subprocess
import csv
import os
from sys import executable
import pandas as pd

# ==========================================
# 0. SETUP FOLDER BUILD
# ==========================================
BUILD_DIR = "PR5_Build"
os.makedirs(BUILD_DIR, exist_ok=True)
print(f"[*] Tahap 0: Folder '{BUILD_DIR}' siap digunakan.")

# Definisi Path
LL_C = os.path.join(BUILD_DIR, "linkedlist.c")
ARR_C = os.path.join(BUILD_DIR, "array.c")
LL_BIN = os.path.join(BUILD_DIR, "run_ll")
ARR_BIN = os.path.join(BUILD_DIR, "run_arr")
LL_CSV = os.path.join(BUILD_DIR, "metrics_ll.csv")
ARR_CSV = os.path.join(BUILD_DIR, "metrics_arr.csv")

# ==========================================
# 1. TULIS FILE C KE DALAM FOLDER BUILD
# ==========================================
ll_code = """#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

typedef struct { int id; char species[50]; float gps_lat; float gps_lon; int age; float diameter; } Tree;
typedef struct Node { Tree data; struct Node* next; } Node;

void addTree(Node** head, Tree newTree) {
    Node* newNode = (Node*)malloc(sizeof(Node));
    newNode->data = newTree; newNode->next = NULL;
    if (*head == NULL) { *head = newNode; return; }
    Node* temp = *head;
    while (temp->next != NULL) temp = temp->next;
    temp->next = newNode;
}

void viewTrees(Node* head) {
    Node* temp = head;
    volatile int dummy = 0;
    while (temp != NULL) { dummy += temp->data.id; temp = temp->next; }
}

void deleteTree(Node** head, int targetID) {
    if (*head == NULL) return;
    Node* temp = *head; Node* prev = NULL;
    if (temp != NULL && temp->data.id == targetID) { *head = temp->next; free(temp); return; }
    while (temp != NULL && temp->data.id != targetID) { prev = temp; temp = temp->next; }
    if (temp == NULL) return;
    prev->next = temp->next; free(temp);
}

int main(int argc, char *argv[]) {
    if (argc < 2) return 1;
    int N = atoi(argv[1]);
    Tree dummyTree = {0, "Pinus", -6.9, 107.6, 10, 45.0};
    Node* head = NULL;

    clock_t start = clock();
    for (int i = 0; i < N; i++) { dummyTree.id = i; addTree(&head, dummyTree); }
    double time_add = ((double)(clock() - start)) / CLOCKS_PER_SEC;

    start = clock();
    viewTrees(head);
    double time_view = ((double)(clock() - start)) / CLOCKS_PER_SEC;

    start = clock();
    deleteTree(&head, N/2);
    double time_delete = ((double)(clock() - start)) / CLOCKS_PER_SEC;

    printf("%f,%f,%f\\n", time_add, time_view, time_delete);
    return 0;
}
"""

arr_code = """#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

typedef struct { int id; char species[50]; float gps_lat; float gps_lon; int age; float diameter; } Tree;
typedef struct { Tree* data; int count; int capacity; } TreeArray;

void initArray(TreeArray* arr, int cap) {
    arr->capacity = cap; arr->count = 0;
    arr->data = (Tree*)malloc(arr->capacity * sizeof(Tree));
}

void addTree(TreeArray* arr, Tree newTree) {
    if (arr->count == arr->capacity) {
        arr->capacity *= 2;
        arr->data = (Tree*)realloc(arr->data, arr->capacity * sizeof(Tree));
    }
    arr->data[arr->count++] = newTree;
}

void viewTrees(TreeArray* arr) {
    volatile int dummy = 0; 
    for (int i = 0; i < arr->count; i++) { dummy += arr->data[i].id; }
}

void deleteTree(TreeArray* arr, int targetID) {
    int foundIndex = -1;
    for (int i = 0; i < arr->count; i++) {
        if (arr->data[i].id == targetID) { foundIndex = i; break; }
    }
    if (foundIndex == -1) return;
    for (int i = foundIndex; i < arr->count - 1; i++) { arr->data[i] = arr->data[i + 1]; }
    arr->count--;
}

int main(int argc, char *argv[]) {
    if (argc < 2) return 1;
    int N = atoi(argv[1]);
    Tree dummyTree = {0, "Pinus", -6.9, 107.6, 10, 45.0};
    TreeArray arr; initArray(&arr, 2);

    clock_t start = clock();
    for (int i = 0; i < N; i++) { dummyTree.id = i; addTree(&arr, dummyTree); }
    double time_add = ((double)(clock() - start)) / CLOCKS_PER_SEC;

    start = clock();
    viewTrees(&arr);
    double time_view = ((double)(clock() - start)) / CLOCKS_PER_SEC;

    start = clock();
    deleteTree(&arr, N/2);
    double time_delete = ((double)(clock() - start)) / CLOCKS_PER_SEC;

    printf("%f,%f,%f\\n", time_add, time_view, time_delete);
    return 0;
}
"""

with open(LL_C, "w") as f:
    f.write(ll_code)
with open(ARR_C, "w") as f:
    f.write(arr_code)

print("[*] Tahap 1: File C fisik berhasil dibuat di dalam folder.")

# ==========================================
# 2. COMPILE C KE BINARY
# ==========================================
try:
    subprocess.run(["gcc", "-O3", LL_C, "-o", LL_BIN], check=True)
    subprocess.run(["gcc", "-O3", ARR_C, "-o", ARR_BIN], check=True)
    print("[*] Tahap 2: Kompilasi GCC berhasil!")
except Exception as e:
    print(f"[!] Error Kompilasi: {e}")

# ==========================================
# 3. BENCHMARK ENGINE
# ==========================================
print("[*] Tahap 3: Menjalankan Testbench...")
DATA_SIZES = {'Small': 1000, 'Medium': 50000, 'Large': 200000}
NODE_SIZE_BYTES = 72
TREE_SIZE_BYTES = 64

def run_benchmark(executable, size_bytes):
    results = []
    for label, n in DATA_SIZES.items():
        # Tambahin prefix './' biar dieksekusi benar di Unix
        process = subprocess.run(['./' + executable, str(n)], capture_output=True, text=True)
        t_add, t_view, t_del = map(float, process.stdout.strip().split(','))
        
        mem_mb = (n * size_bytes) / (1024 * 1024)
        tp_add = n / t_add if t_add > 0 else float('inf')
        tp_view = n / t_view if t_view > 0 else float('inf')
        
        results.append({
            'Data_Size': label,
            'N': n,
            'Mem_Usage (MB)': round(mem_mb, 4),
            'Time_Add (s)': round(t_add, 6),
            'TP_Add (ops/s)': round(tp_add, 2),
            'Time_View (s)': round(t_view, 6),
            'TP_View (ops/s)': round(tp_view, 2),
            'Time_Delete (s)': round(t_del, 8)
        })
    return results

# Eksekusi dan simpan CSV ke dalam folder build
ll_data = run_benchmark(LL_BIN, NODE_SIZE_BYTES)
arr_data = run_benchmark(ARR_BIN, TREE_SIZE_BYTES)

pd.DataFrame(ll_data).to_csv(LL_CSV, index=False)
pd.DataFrame(arr_data).to_csv(ARR_CSV, index=False)

# ==========================================
# 4. TAMPILKAN HASIL
# ==========================================
print("\n--- HASIL METRIK: LINKED LIST ---")
display(pd.read_csv(LL_CSV))

print("\n--- HASIL METRIK: DYNAMIC ARRAY ---")
display(pd.read_csv(ARR_CSV))

[*] Tahap 0: Folder 'PR5_Build' siap digunakan.
[*] Tahap 1: File C fisik berhasil dibuat di dalam folder.
[*] Tahap 2: Kompilasi GCC berhasil!
[*] Tahap 3: Menjalankan Testbench...

--- HASIL METRIK: LINKED LIST ---


,Data_Size,N,Mem_Usage (MB),Time_Add (s),TP_Add (ops/s),Time_View (s),TP_View (ops/s),Time_Delete (s)
0,Small,1000,0.0687,0.000933,1071811.36,0.000003,3.333333e+08,0.000001
1,Medium,50000,3.4332,1.416620,35295.28,0.000062,8.064516e+08,0.000031
2,Large,200000,13.7329,24.375080,8205.10,0.000278,7.194245e+08,0.000146



--- HASIL METRIK: DYNAMIC ARRAY ---


,Data_Size,N,Mem_Usage (MB),Time_Add (s),TP_Add (ops/s),Time_View (s),TP_View (ops/s),Time_Delete (s)
0,Small,1000,0.0610,0.000045,22222222.22,0.000004,2.500000e+08,0.000003
1,Medium,50000,3.0518,0.000835,59880239.52,0.000121,4.132231e+08,0.000126
2,Large,200000,12.2070,0.003898,51308363.26,0.001637,1.221747e+08,0.001096
